---
title: "Data Preparation"
number-sections: false
---

In Task 1, you worked with the file `starter_dataset.xlsx`.
This dataset contained:

* the **SMILES string** in the first column
* a set of **molecular descriptors** in the following columns
* the last three columns as the **target variables**: `homo`, `lumo`, and `stability`

The starter dataset included **80 molecules** and was provided to help you build your first models quickly.

Now, your task is to replicate this dataset format for the **full set of molecules** in HOPV.

### Steps

1. Load `HOPV_stability.csv`
2. Load `HOPV_homolumo_processed.csv`
3. Merge the two datasets
4. Calculate molecular descriptors using **RDKit**
5. Create a DataFrame with the same structure as `starter_dataset.xlsx`:

   * First column: SMILES string
   * Middle columns: molecular descriptors
   * Last three columns: `homo`, `lumo`, `stability`

### Expected Output

* Instead of 80 molecules, your final DataFrame should contain **347 molecules**.
* The `starter_dataset.xlsx` you used earlier was created by following the same process and then sampling 80 molecules from the full dataset.


**Missing packages?** This notebook uses `rdkit`, which may not be installed in your environment.

- **Google Colab:** run the cell below. It installs the packages only when it detects Colab, so it does nothing when you run it locally.
- **Locally (uv):** run `uv add rdkit` in a terminal at the project root, then restart the kernel.

In [ ]:
# Installs the missing packages on Google Colab only; does nothing locally.
import sys

if "google.colab" in sys.modules:
    %pip install -q rdkit

In [ ]:
# IMPORT WHAT YOU NEED HERE

<mark>Your Turn</mark>

- Read `HOPV_stability.csv` into `stability_df`
- Read `HOPV_homolumo_processed.csv` into `homolumo_df`


In [ ]:
# YOUR CODE HERE

In [ ]:
# SOLUTION
stability_df.head()

We have 350 SMILES strings in both dataframes. One dataframe contains the HOMO and LUMO values, while the other contains the stability values. Our goal is to combine them into a single dataframe using the `merge` method.

From the data understanding phase, we know that some SMILES appear more than once. These duplicates must be resolved before we can safely perform the merge.

<mark>Your Turn</mark>

Count the number of unique SMILES in both `stability_df` and `homolumo_df`, and check that they match one-to-one.

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

Drop the duplicates with the `drop_duplicates` method. Use the parameters

```python
drop_duplicates(subset="smiles", keep="first")
```

You should end up with 347 SMILES in both DataFrames.


This will keep the first occurrence of each SMILES and remove the rest. Do you notice any potential issues with relying on this approach?

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

Merge the DataFrames with `homolumo_df` as the left dataframe and `stability_df` as the right dataframe, so that the HOMO and LUMO columns appear before the stability column in the result. Store the merged DataFrame in a variable named `df`. Verify you have 347 SMILES in `df`.

In [ ]:
# YOUR CODE HERE

## RDKit

Now we are ready to calculate molecular descriptors using RDKit. This can be done in just a few lines of code. For this demo, let us calculate `MolWt` which is the molecular weight, sum of the atomic weights of all atoms in the molecule, of the molecule butene.

https://en.wikipedia.org/wiki/Butene

Butene's chemical formula is $\mathrm{C_{4}H_{8}}$ and the SMILES string representation is `CCC=C`. Note that the SMILES string does not explicitly include hydrogen atoms, because RDKit (and most cheminformatics tools) automatically adds the correct number of implicit hydrogens to each atom to satisfy its valence.

In [ ]:
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors

In [ ]:
# Step 1: choose the descriptors to calculate
descriptor_names = ["MolWt"] # you can add more descriptors to the list
descriptor_names

In [ ]:
# Step 2: create the calculator object based on the descriptor names
calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

In [ ]:
# Step 3: create a molecule object from a SMILES string
smiles_str = "CCC=C"
mol = Chem.MolFromSmiles(smiles_str)
mol

In [ ]:
# Step 4: calculate descriptors for this molecule
features = calc.CalcDescriptors(mol)

print("Descriptor names:", descriptor_names)
print("Values:", features)

Butene contains 4 Carbon atoms and 8 Hydrogen atoms. The atomic weight of [Carbon](https://en.wikipedia.org/wiki/Carbon) is ~12.011 and [Hydrogen](https://en.wikipedia.org/wiki/Hydrogen) is ~1.008. We can calculate the expected MolWt ourselves:

In [ ]:
# Expected MolWt
4*12.011 + 8*1.008

We can get all the built-in descriptors names as follows:

In [ ]:
descriptor_names = [x[0] for x in Descriptors._descList]
print(f"There are {len(descriptor_names)} descriptors.")

take a look at the first 10

In [ ]:
descriptor_names[:10]

<mark>Your Turn</mark>

Calculate all available molecular descriptors for butene (`CCC=C`). Store the results in a pandas DataFrame, using the descriptor names as the column headers and the calculated values as the row. **Do not use any loops**.

<figure style="text-align: center;">
    <img src='notebook_images/butene.png' alt="CRISP-DM" width="80%">
    <figcaption><em>Expected output</a>.</em></figcaption>
</figure>

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

Write a function called `compute_descriptors` that takes a SMILES string as input, and returns a tuple containing all 217 descriptor values. Note that the number of molecular descriptors depends on the RDKit version.

In [ ]:
def compute_descriptors(smiles_str):
    """
    WRITE A DESCRIPTION

    Parameters
    ----------
    smiles_str : str
        The SMILES string representation of a molecule.

    Returns
    -------
    tuple
        A tuple containing the values of all available molecular descriptors
        (217 descriptors in total, from RDKit descriptor list).
    """
    # YOUR CODE HERE
    features = None
    return features

<mark>Your Turn</mark>

Apply the `compute_descriptors` function to the smiles column in `df` to calculate molecular descriptors of all the molecules in `df`. Save the results into a DataFrame called `descriptors_df` with proper column headings. **Do not use any loops**.

<figure style="text-align: center;">
    <img src='notebook_images/descriptors_df.png' alt="CRISP-DM" width="80%">
    <figcaption><em>Expected output</a>.</em></figcaption>
</figure>

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

The `descriptors_df` currently contains only the molecular descriptors. It does **not** include the SMILES strings or the target variables. While SMILES strings are not directly used for modeling, they are essential for keeping track of which molecule each row belongs to. In addition, we need the target columns in order to train a machine learning model.

Create a new dataframe called `xy_df`. The name comes from the standard machine learning notation where `X` represents the input features and `y` represents the targets. This dataframe should bring together both the descriptors (features) and the target variables into one place. Use the following order for the columns:

1. SMILES strings
2. Descriptors
3. Targets: `homo`, `lumo`, `stability`

<figure style="text-align: center;">
    <img src='notebook_images/xy_df.png' alt="CRISP-DM" width="90%">
    <figcaption><em>Expected output</a>.</em></figcaption>
</figure>

In [ ]:
# YOUR CODE HERE

The DataFrame `xy_df` can almost be used as it is to train a machine learning model. But that does not mean it is in optimal form for modeling.

Better processing will give better model performance. In the next iterations, you will process xy_df further to get better models.

<mark>Your Turn</mark>

Save `xy_df` as a csv file to the folder `002_processed_data` with the name `xy_df_hopv.csv`.

In [ ]:
# YOUR CODE HERE